In [ ]:
# ── 第一格：环境准备（D02 封装之后，每本 notebook 开头都是这十几行）──
%load_ext autoreload
%autoreload 2
# 改了 src/ 里的 .py，notebook 不用重启就能用上新代码。
from signal_mysql import database as mysql
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from signal_insight import config, geo, privacy, quality, viz
from signal_insight.viz import MAIN, MAIN_LIGHT, MAIN_DARK, MAIN_MID, GRAY, GRAY_DARK, RED, AMBER, INK, MUTED, FAINT, AREA_COLOR, SEG_COLOR    # 配色常量
viz.apply_theme()                           # 字体、配色、坐标轴样式：一行
con = mysql.connect(config.DB_SCHEMA)       # 路径从包里来：notebook 在哪个文件夹打开都找得到库
DAY = config.ANALYSIS_DAY                   # 分析日期（口径常量）
NB = "D08_标签与口径"
SOURCE = f"{config.SOURCE_NOTE}；脚本：{NB}"    # 每张图左下角的来源行
pd.set_option("display.max_columns", 20)

In [ ]:
from signal_mysql import database as mysql
# 【跟丢了才运行】D6、D7 没做完的同学：把 False 改成 True 再运行，从检查点库复制标准表
CATCH_UP = False
if CATCH_UP:
    with mysql.connect(config.CHECKPOINT_SCHEMA) as ck:
        for t in ["dwd_signal", "dim_cell", "dim_hotspot", "dws_stay", "dws_trip", "dws_night"]:
            mysql.write_frame(mysql.read_frame(f"SELECT * FROM {t}", ck), t, con, if_exists="replace", index=False)
            print("已恢复", t)

In [ ]:
from signal_mysql import database as mysql
from signal_insight import db, dims, dws, pipeline
settings = config.load_settings()
hot = mysql.read_frame("SELECT * FROM dim_hotspot", con).iloc[0]       # 西塔中心（D4）
DAY_START = pd.Timestamp(DAY)                                      # 2018-10-03 00:00

In [ ]:
from signal_mysql import database as mysql
stay = mysql.read_frame("SELECT * FROM dws_stay", con, parse_dates=["start_time", "end_time"])
trip = mysql.read_frame("SELECT * FROM dws_trip", con, parse_dates=["depart_time", "arrive_time"])
night = mysql.read_frame("SELECT * FROM dws_night", con).set_index("imsi")
sig = mysql.read_frame("""SELECT s.imsi, s.ts, c.lng, c.lat FROM dwd_signal s
                     JOIN dim_cell c ON s.lac_cell = c.lac_cell""", con, parse_dates=["ts"])
users = pd.Index(sorted(sig["imsi"].unique()), name="imsi")
print(f"用户 {len(users):,}；停留 {len(stay):,}；出行 {len(trip):,}；有过夜地 {len(night):,}")

In [ ]:
# 这一格在做什么：定义"傍晚访客"——今天所有结论的人群范围。
#
# 为什么判断的是停留的**结束**时刻 ≥ 17:00，而不是开始时刻：
# 下午 3 点到西塔、一直待到晚上 8 点的人，当然算傍晚在场。按开始时刻判断就会把他漏掉。
# 讲台提示：先让学生自己写这一行，大多数人会写 start_time——然后问"下午来、晚上还没走的人呢？"
#
# ⚠ 到这里为止，"500 米"和"17 点"两个数就把人群框死了。换一个数，下面所有结论的分母都变。
# 这正是 8.5 要集中展示的事。
xita = stay[stay["dist_xita_m"] <= 500]
evening = xita[xita["end_time"] >= DAY_START + pd.Timedelta(hours=17)]
visitors = pd.Index(evening["imsi"].unique(), name="imsi")
print(f"有西塔停留的：{xita['imsi'].nunique():,} 人；其中傍晚访客：{len(visitors):,} 人，占全部用户 {len(visitors) / len(users):.1%}")

In [ ]:
# 这一格在做什么：把前几天的表汇总成一张"用户日特征宽表"——**一个人一行**，几十个特征列。
#
# 为什么要有这张表：
#   · 下游（标签、分析、平台）不用每次都去 join 四五张明细表；
#   · 它就是机器学习的标准输入格式（一行一个样本、一列一个特征），下一门课直接拿来用；
#   · 企业里这类表叫"用户画像宽表"或"ADS 用户日表"，是数仓里最常被查的一类。
#
# 写法上注意：index 先用全量用户 users 建好，再把各张表 groupby 的结果**按 index 对齐赋值**。
# 这样没有停留、没有出行的人会自动变成 NaN，最后统一 fillna(0)，不会被悄悄丢掉。
day = pd.DataFrame(index=users)
# 活跃
g = sig.groupby("imsi")
day["n_records"] = g.size()
day["first_seen"] = g["ts"].min().dt.strftime("%H:%M")
day["last_seen"] = g["ts"].max().dt.strftime("%H:%M")
day["active_hours"] = sig["ts"].dt.hour.groupby(sig["imsi"]).nunique()           # 向量化，不用 apply（D5）
# 停留与出行
gs = stay.groupby("imsi")
day["n_stays"] = gs.size()
day["stay_total_min"] = gs["duration_min"].sum()
day["max_stay_min"] = gs["duration_min"].max()
day["n_trips"] = trip.groupby("imsi").size()
day["trip_total_km"] = trip.groupby("imsi")["dist_m"].sum() / 1000
# 回转半径（gyration）：一个人当天活动范围的大小。算法分两步——
#   ① 先求**按停留时长加权**的重心（待得久的地方权重大）；
#   ② 再求各停留点到重心距离的加权均方根。
# 加权而不是简单平均，是因为"在家待 10 小时"和"路过某地停 30 分钟"对活动范围的意义不同。
# 这个指标在人口流动研究里是标准做法，D8.10 会用它比较各客群。
w = stay["duration_min"]
c_lng = (stay["lng"] * w).groupby(stay["imsi"]).sum() / w.groupby(stay["imsi"]).sum()    # 按停留时长加权的重心
c_lat = (stay["lat"] * w).groupby(stay["imsi"]).sum() / w.groupby(stay["imsi"]).sum()
d_c = geo.haversine(stay["lat"], stay["lng"], stay["imsi"].map(c_lat), stay["imsi"].map(c_lng))
day["gyration_km"] = np.sqrt((w * d_c ** 2).groupby(stay["imsi"]).sum() / w.groupby(stay["imsi"]).sum()) / 1000
# 过夜地
day["night_district"] = night["night_district"]
day["night_dist_xita_km"] = night["night_dist_xita_m"] / 1000
# 西塔相关
gx = xita.groupby("imsi")
day["xita_stay_min"] = gx["duration_min"].sum()
day["xita_arrive_hour"] = (gx["start_time"].min() - DAY_START).dt.total_seconds() / 3600    # 首次到达西塔，几点（小数）
day["is_evening_visitor"] = day.index.isin(visitors)
last_leave = gx["end_time"].max()
day["xita_after_23"] = day.index.isin(last_leave[last_leave >= DAY_START + pd.Timedelta(hours=23)].index)
day["visit_taiyuan"] = day.index.isin(stay.loc[stay["aoi"] == "太原街商圈", "imsi"])
day["visit_zhongjie"] = day.index.isin(stay.loc[stay["aoi"] == "中街商圈", "imsi"])
fill0 = ["n_stays", "stay_total_min", "max_stay_min", "n_trips", "trip_total_km", "xita_stay_min"]
day[fill0] = day[fill0].fillna(0)                                             # 没有停留/出行的记 0
day = day.round(2)
print(day.shape)
day[day["is_evening_visitor"]].head(3).rename(index=privacy.mask_id).T    # 展示前脱敏

In [ ]:
from signal_mysql import database as mysql
mysql.write_frame(day.reset_index(), "dws_user_day", con, if_exists="replace", index=False)
chk = mysql.read_frame("SELECT COUNT(*) AS 行数, SUM(is_evening_visitor) AS 傍晚访客 FROM dws_user_day", con)
chk

In [ ]:
# 对账
assert len(day) == 5_644 and chk.loc[0, "傍晚访客"] == len(visitors)
print(f"✅ 对账通过：dws_user_day 5,644 行，傍晚访客 {len(visitors):,} 人")

In [ ]:
from signal_mysql import database as mysql
# ════ 客群标签：注意判断的顺序 ════
# np.select 是**从前到后**取第一个成立的条件，所以下面四行的顺序本身就是业务规则：
#   ① 过夜地识别不出来  → "过夜地不明"（放在最前面：先承认不知道，再谈分类）
#   ② 过夜地在沈阳市外  → "市外来访"
#   ③ 过夜地离西塔 < 1.5 公里 → "周边居民"
#   ④ 其余 → "市内来访"
#
# 讲台提示：把 ② 和 ③ 对调再跑一次，数字就会变——市外但碰巧离西塔近的人会被算成周边居民。
# **让学生亲眼看到"换个顺序，结论就变"**，比讲十遍"口径很重要"管用。
#
# ⚠ "过夜地不明"不是一类人，是"我们不知道"。所以它在所有图里固定是灰色（SEG_COLOR），
# 而且绝不能被当成"本地人"或"外地人"去解释。
dist = day["night_dist_xita_km"]
day["segment"] = np.select(
    [dist.isna(), day["night_district"] == "市外", dist < 1.5],
    ["过夜地不明", "市外来访", "周边居民"],
    default="市内来访")
mysql.write_frame(day.reset_index(), "dws_user_day", con, if_exists="replace", index=False)        # 带上标签重新写回
v = day[day["is_evening_visitor"]]
seg_tbl = v.groupby("segment").agg(n_users=("segment", "size"),
                                   xita_stay_min_p50=("xita_stay_min", "median"),
                                   first_arrive_hour_p50=("xita_arrive_hour", "median"))
seg_tbl["share"] = seg_tbl["n_users"] / seg_tbl["n_users"].sum()
seg_tbl = seg_tbl.sort_values("n_users")
mysql.write_frame(seg_tbl, "ads_visitor_segment", con, if_exists="replace", index=False)
seg_tbl

In [ ]:
# ── 出图 D8_01：横向条形图 ───────────────────────────────────────────────
# 为什么条形不用饼图：这里有 4 类，饼图勉强能用，但读者要比较的是"周边居民比市内来访多多少"——
# 比角度远不如比长度准。规范里写的"类别超过 5 块不用饼图"是底线，**能用条形就别用饼**是习惯。
# 规范落在哪几行：
#   · color=[SEG_COLOR[k]...] —— 客群用**固定映射**，和 D8_04、D8_06、平台上的颜色完全一致
#   · "过夜地不明"是灰色，副标题特意写"不是一类人" —— 不能让读者把它当成一个客群去解读
#   · annotate 那段是业务注释：图上直接抛出"外地人只有 1%？"的疑问，
#     把读者引向下一张图。**一张图只讲一件事，讲不完的交给下一张**
# 标题里的数字：s_[top_] / len(v)（占比最高的客群）、s_["周边居民"] / len(v)。
s_ = v["segment"].value_counts()
order = s_.sort_values().index
fig, ax = plt.subplots(figsize=(10, 5.6), layout="constrained")
bars = ax.barh(order, s_[order], color=[SEG_COLOR[k] for k in order], height=0.6)    # 客群按固定颜色
ax.bar_label(bars, labels=[f"{s_[k] / len(v):.0%}（{s_[k]:,} 人）" for k in order], padding=4, fontsize=10, color=INK)
ax.set_xlim(0, s_.max() * 1.35); ax.set_xticks([]); ax.grid(False); ax.tick_params(axis="y", labelsize=10.5, labelcolor=INK)
i_ = list(order).index("市外来访")
ax.annotate("国庆假期的商圈，外地人只有 1%？\n下一张图（D8_04）换个口径再看", xy=(s_["市外来访"], i_),
            xytext=(s_.max() * 0.55, i_ + 0.1), fontsize=9.5, color=MUTED, va="center",
            arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.8))             # 业务注释（手册 7.2 部件 5）
top_ = s_.idxmax()
title = f"傍晚访客里 {s_[top_] / len(v):.0%} 是{top_}，{s_['周边居民'] / len(v):.0%} 是周边居民"
viz.finish(fig, title,
           f"{len(v):,} 个傍晚访客按过夜地分类：周边居民 = 过夜地离西塔 1.5 公里内；灰色 = 识别不出过夜地（凌晨信令太少），不是一类人；口径：dws_user_day",
           SOURCE, "D8_01_傍晚访客客群.png")

In [ ]:
from signal_mysql import database as mysql
sig_d = mysql.read_frame("""SELECT s.imsi, s.ts, c.lng, c.lat, c.district FROM dwd_signal s
                       JOIN dim_cell c ON s.lac_cell = c.lac_cell""", con, parse_dates=["ts"])
# ════ 本日最该讲的推理过程：数字反常 → 去查口径 ════
# 现象：8.3 算出来"市外来访"只占 1%。国庆假期的核心商圈，外地人只有 1%？不合常理。
# 追查：8.3 判断"是不是外地人"只看了**过夜地**。可是当天白天才从外地进城的人，
#       凌晨还在外地，根本不在我们的观测范围里，于是过夜地识别不出 → 被归进"过夜地不明"。
# 修正：加一条**独立证据**——当天第一条信令在市域外、之后进入市域，就是外地来客。
#
# 下面三行就是这条规则。kind="stable" 保证时间戳相同的记录不会被打乱顺序，
# 否则"当天第一条信令"可能每次跑出来都不一样。
#
# 讲台提示：企业里"数字反常"的第一反应应该是查口径，不是改数字。这一卡就是现场演示。
sig_d = sig_d.sort_values(["imsi", "ts"], kind="stable")
first = sig_d.groupby("imsi").first()                                    # 每个人当天的第一条信令
entry = sig_d[sig_d["district"] != "市外"].groupby("imsi").first()       # 第一次出现在市域内的那一条
inbound = first.index[first["district"] == "市外"].intersection(entry.index)
day["inbound"] = day.index.isin(inbound)
day["segment_v2"] = np.where(day["inbound"] | (day["segment"] == "市外来访"), "外地来客", day["segment"])
mysql.write_frame(day.reset_index(), "dws_user_day", con, if_exists="replace", index=False)      # 宽表多两列，重新写回
ev = day[day["is_evening_visitor"]]
order = ["周边居民", "市内来访", "外地来客", "过夜地不明"]
old = ev["segment"].replace({"市外来访": "外地来客"}).value_counts().reindex(order, fill_value=0)
new = ev["segment_v2"].value_counts().reindex(order, fill_value=0)
moved = int((ev["inbound"] & (ev["segment"] == "过夜地不明")).sum())
print(f"当天从市外进城 {len(inbound):,} 人，其中 {day.loc[day['inbound'], 'is_evening_visitor'].mean():.0%} 成了傍晚访客；"
      f"原来被归进“过夜地不明”的有 {moved} 人")
pd.DataFrame({"原口径（按过夜地）": old, "修正后（加入入城轨迹）": new})

In [ ]:
# 对账
assert len(inbound) == 640 and new["外地来客"] == 473 and moved == 436
print(f"✅ 对账通过：入城 640 人；傍晚访客里的外地来客从 {old['外地来客']} 人修正为 473 人（{473 / len(ev):.1%}）")

In [ ]:
# ── 出图 D8_04：桑基图（流向图）───────────────────────────────────────
# 为什么用桑基图：要回答的问题是"改了口径之后，人是**从哪一类流到哪一类**的"。
# 两张并排的条形图只能看出前后总数变了，看不出是谁变成了谁。桑基图的带子直接给出答案。
# 什么时候**不能**用桑基：类别之间不是"同一批人的重新分配"时（比如 D10_01 的联动，
# 一个人可以同时属于多类，合计超过 100%），桑基的带宽就没有意义——D10_01 的副标题里
# 特意写了"一人可属多类，比例不能相加（所以不用桑基图）"。
# 规范落在哪几行：
#   · 只有"改判为外地来客"的带子上橙色，其余灰色 —— 一种颜色强调一件事
#   · 左右两侧都写了类别名 + 人数 + 占比 —— 桑基图最容易让人只看形状不看量，必须标数
#   · 左右两端的说明文字（"按过夜地贴标签" / "加上当天入城轨迹"）——
#     桑基图不写清左右各是什么口径，读者无从理解
# 标题里的数字：old["外地来客"]（旧口径）、new["外地来客"]（新口径）及其占比。
# 下面 tt 那一行的 (1-cos)/2 是把直线插值变成 S 形曲线，让带子平滑，纯样式。
from matplotlib.patches import Rectangle
order = ["周边居民", "市内来访", "外地来客", "过夜地不明"]
ct = pd.crosstab(ev["segment"].replace({"市外来访": "外地来客"}), ev["segment_v2"]).reindex(index=order, columns=order, fill_value=0)
n_ = len(ev)
fig, ax = plt.subplots(figsize=(10, 5.6), layout="constrained")
# 桑基图：左旧口径、右新口径，带子宽度 = 人数；只给改判为外地来客的带子上色
gap_ = n_ * 0.035; xL, xR, w = 0.0, 1.0, 0.045
def stack(tot):
    pos, y = {}, 0
    for kk in order:
        pos[kk] = [y, y + tot[kk]]; y += tot[kk] + gap_
    return pos
L, R = stack(ct.sum(axis=1)), stack(ct.sum(axis=0))    # pandas 4 起位置参数会报错
lo, ro = {kk: L[kk][0] for kk in order}, {kk: R[kk][0] for kk in order}
xs = np.linspace(xL + w, xR - w, 60); tt = (1 - np.cos(np.pi * (xs - xL - w) / (xR - xL - 2 * w))) / 2
for a in order:
    for b in order:
        val = ct.loc[a, b]
        if val == 0:
            continue
        top_ = lo[a] + (ro[b] - lo[a]) * tt; bot = lo[a] + val + (ro[b] - lo[a]) * tt
        hi = b == "外地来客" and a != "外地来客"
        ax.fill_between(xs, top_, bot, color=SEG_COLOR["外地来客"] if hi else GRAY, alpha=0.75 if hi else 0.35, lw=0)
        lo[a] += val; ro[b] += val
for side, pos, x_ in [("L", L, xL), ("R", R, xR - w)]:
    for kk in order:
        y0_, y1_ = pos[kk]
        ax.add_patch(Rectangle((x_, y0_), w, y1_ - y0_, color=SEG_COLOR[kk], lw=0))
        txt = f"{kk}   {int(y1_ - y0_):,}（{(y1_ - y0_) / n_:.0%}）"
        ax.text(x_ - 0.012 if side == "L" else x_ + w + 0.012, (y0_ + y1_) / 2, txt, ha="right" if side == "L" else "left",
                va="center", fontsize=9.5, color=INK, fontweight="bold" if (side == "R" and kk == "外地来客") else "normal")
ax.text(0.5, R["外地来客"][0] - gap_ * 1.1, f"{moved} 人从“过夜地不明”改判为外地来客", ha="center", va="bottom",
        fontsize=9.5, color="#B45F06", fontweight="bold")
ax.text(xL + w / 2, -gap_ * 1.6, "按过夜地贴标签（8.3 的口径）", ha="center", fontsize=10, color=MUTED)
ax.text(xR - w / 2, -gap_ * 1.6, "加上当天入城轨迹（修正后）", ha="center", fontsize=10, color=MUTED)
ax.set_xlim(-0.36, 1.36); ax.set_ylim(R["过夜地不明"][1] + gap_ * 0.5, -gap_ * 2.6); ax.axis("off")
title = f"按过夜地只找到 {old['外地来客']} 个外地来客，看入城轨迹是 {new['外地来客']} 人（{new['外地来客'] / new.sum():.0%}）"
viz.finish(fig, title,
           f"{n_:,} 个傍晚访客从旧口径（左）流向新口径（右），橙色 = 改判为外地来客的人；\n外地来客 = 当天第一条信令在沈阳市域外、之后进入市域（或过夜地在市外）；口径：dws_user_day、dwd_signal × dim_cell",
           SOURCE, "D8_04_客群口径修正.png")

In [ ]:
from signal_mysql import database as mysql
DIRECTION = {"苏家屯区": "南", "辽中区": "西南", "沈北新区": "北", "浑南区": "东南", "新民市": "西北",
             "于洪区": "西", "大东区": "东北", "法库县": "北", "康平县": "北", "铁西区": "西", "皇姑区": "北"}
e = entry.loc[inbound]
entry_hour = e["ts"].dt.hour
share_11_16 = entry_hour.between(11, 16).mean()
entry_district = e["district"].value_counts()
# 下面这段是"推测"，不是结论：进城后 2 小时内到过火车站附近 ≠ 一定坐火车来
# （住在火车站附近的人也会被算进来）。所以它只出现在**副标题**里，而且用了"推测"两个字，
# 没有进标题，也没有进报告结论。
# 讲台提示：证据强度决定它能出现在哪里——强证据进标题，弱证据进副标题或附录。
# 进城后 2 小时内到过沈阳站、沈阳北站 600 米内的，推测坐火车来
tr = mysql.read_frame("SELECT * FROM ods_poi_transport", con)
st_ = tr[tr["name"].isin(["沈阳站(地铁站)", "沈阳北站(地铁站)"])]
s2 = sig_d[sig_d["imsi"].isin(inbound)].join(e["ts"].rename("ent"), on="imsi")
w_ = s2[(s2["ts"] >= s2["ent"]) & (s2["ts"] <= s2["ent"] + pd.Timedelta(hours=2))]
near = np.zeros(len(w_), dtype=bool)
for _, r_ in st_.iterrows():
    near |= geo.haversine(w_["lat"], w_["lng"], r_["lat"], r_["lng"]) <= 600
rail = w_.loc[near, "imsi"].nunique()
hc = entry_hour.value_counts().reindex(range(24), fill_value=0)
top6 = entry_district.head(6)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 5.6), layout="constrained", width_ratios=[1.45, 1])
a1.axvspan(10.5, 16.5, color=MAIN_LIGHT, lw=0)
a1.plot(hc.index, hc.values, color=MAIN, lw=2.2, marker="o", ms=3.5)              # 时间序列用折线
a1.text(13.5, hc.max() * 1.06, f"11–16 点进城 {share_11_16:.0%}", ha="center", fontsize=9.5, color=MAIN, fontweight="bold")
a1.set_xlim(-0.5, 23.5); a1.set_ylim(0, hc.max() * 1.18)
a1.set_xticks(range(0, 24, 3), [f"{x} 点" for x in range(0, 24, 3)])
a1.set_title("第一次出现在沈阳市域内的时刻（人）", loc="left", fontsize=10.5, color=INK, fontweight="bold")
labs = [f"{k}（{DIRECTION.get(k, '')}）" for k in top6.index] + ["其他区县"]
vals = list(top6.values) + [entry_district.iloc[6:].sum()]
bars = a2.barh(labs[::-1], vals[::-1], color=[GRAY] * 5 + [MAIN, MAIN], height=0.62)
a2.bar_label(bars, labels=[f"{v:,}" for v in vals[::-1]], padding=3, fontsize=9.5, color=INK)
a2.set_xlim(0, max(vals) * 1.25); a2.set_xticks([]); a2.grid(False); a2.tick_params(axis="y", labelcolor=INK)
a2.set_title("进城时所在区县（相对市中心的方向）", loc="left", fontsize=10.5, color=INK, fontweight="bold")
title = f"{len(inbound):,} 人当天从市外进城，{share_11_16:.0%} 在 11–16 点；从{top6.index[0]}、{top6.index[1]}方向进入最多"
viz.finish(fig, title,
           f"入城来客 = 当天第一条信令在沈阳市域外、之后进入市域的样本用户；进入时刻取第一条市域内信令；\n其中 {rail} 人进城后 2 小时内到过沈阳站或沈阳北站附近，推测坐火车来；口径：dwd_signal × dim_cell",
           SOURCE, "D8_05_入城时刻与方向.png")

In [ ]:
from signal_mysql import database as mysql
# 包里的版本和 notebook 里一步步建的宽表对账
sig_day = mysql.read_frame("""SELECT s.imsi, s.ts, s.lac_cell, c.lng, c.lat, c.district FROM dwd_signal s
                         JOIN dim_cell c ON s.lac_cell = c.lac_cell""", con, parse_dates=["ts"])
user_day = dws.build_user_day(sig_day, stay, trip, night.reset_index(), DAY, settings)
mine = day.reset_index()[user_day.columns]
pd.testing.assert_frame_equal(user_day, mine, check_dtype=False)
db.replace_table(con, user_day, "dws_user_day")
user_day[user_day["is_evening_visitor"]]["segment_v2"].value_counts()

In [ ]:
# 对账：修正后的客群人数（D08 需求卡 8.4）
seg_v2 = user_day[user_day["is_evening_visitor"]]["segment_v2"].value_counts()
assert len(user_day) == 5_644 and seg_v2.to_dict() == {"市内来访": 1_283, "过夜地不明": 807, "周边居民": 747, "外地来客": 473}
print("✅ 对账通过：dws.build_user_day 与 notebook 版本逐列一致；外地来客 473 人")
# 流水线多了一步：dws_user_day（整表替换）
log = pipeline.run_day()
log[["step", "rows_in", "rows_out", "elapsed_sec"]].tail(4)

In [ ]:
from signal_mysql import database as mysql
# ════ 整门课最该让学生记住的一格 ════
# 这一格在做什么：用两个维度（离西塔多近 × 待多久）组合出 12 种"到访人数"的算法，各算一遍。
#
# 两个维度各自都很合理：
#   · 半径 300 / 500 / 1000 米 —— "到访"是指进了商圈，还是路过附近？
#   · 任意一条信令 / 停留 ≥ 30 / 60 / 120 分钟 —— 路过算不算到访？
# 组合起来，结果从 1,836 到 5,282 人，**差 2.9 倍，全都叫"西塔到访人数"**。
#
# 讲台提示：**先让学生猜"最大和最小差几倍"**，写在白板上，再跑这一格。
# 猜 1.5 倍的最多，答案是 2.9 倍。这个落差是本卡全部的教学价值所在。
#
# 结论不是"哪个口径对"——**都对**，只是回答的不是同一个问题。
# 企业里真正要做的是：选一个、写进口径表（dim_metric）、所有人一直用它。
rows = []
for r in [300, 500, 1000]:
    near = sig[geo.haversine(sig["lat"], sig["lng"], hot["lat"], hot["lng"]) <= r]
    rows.append({"radius": f"{r} 米", "definition": "任意一条信令", "n_users": near["imsi"].nunique()})
    for t_min in [30, 60, 120]:
        n = stay[(stay["dist_xita_m"] <= r) & (stay["duration_min"] >= t_min)]["imsi"].nunique()
        rows.append({"radius": f"{r} 米", "definition": f"停留 ≥ {t_min} 分钟", "n_users": n})
caliber = pd.DataFrame(rows)
mysql.write_frame(caliber, "ads_caliber_visitors", con, if_exists="replace", index=False)
caliber.pivot(index="radius", columns="definition", values="n_users").loc[["300 米", "500 米", "1000 米"]]

In [ ]:
from signal_mysql import database as mysql
# ── 出图 D8_02：数值矩阵（热力表）─────────────────────────────────────
# 为什么用矩阵而不是分组条形图：两个维度（半径 × 时长）都要同时看见，
# 分组条形图把其中一维压成颜色或分组，读者还得在脑子里还原成二维。
# 矩阵一眼就能看出"往右走变少、往下走变多"两个方向的规律。
# 规范落在哪几行：
#   · 每个格子里**直接写数字**（千分位）—— 热力图只给颜色，读者读不出具体值；
#     颜色在这里只负责引导视线，数字才是内容
#   · 文字颜色按格子深浅在白 / 墨之间切换 —— 深底上写黑字会看不清
#   · 黑框标"本项目采用"，**不用红框** —— 红色只留给警示，这里是中性的"我们选了这个"
#   · cmap="Blues" 单色渐变 —— 不用彩虹色阶
# 标题里的数字：lo_、hi_（矩阵的最小值和最大值）和 hi_ / lo_（倍数），全部由代码算。
c = mysql.read_frame("SELECT * FROM ads_caliber_visitors", con)
cols = ["任意一条信令", "停留 ≥ 30 分钟", "停留 ≥ 60 分钟", "停留 ≥ 120 分钟"]
mat = c.pivot(index="radius", columns="definition", values="n_users").loc[["300 米", "500 米", "1000 米"], cols]
fig, ax = plt.subplots(figsize=(10, 5.6))
ax.imshow(mat.values, cmap="Blues", aspect="auto", vmin=0, vmax=mat.values.max() * 1.15)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        val = mat.values[i, j]
        ax.text(j, i, f"{val:,}", ha="center", va="center", fontsize=13,
                color="white" if val > mat.values.max() * 0.6 else INK, fontweight="bold")
ax.add_patch(plt.Rectangle((0.5, 0.5), 1, 1, fill=False, ec=INK, lw=2.5))      # 本项目采用的口径（红色只留给警示）
ax.text(1, 0.6, "本项目采用", ha="center", va="top", fontsize=9, color="white")
ax.set_xticks(range(4), ["任意一条信令", "停留 ≥ 30 分钟", "停留 ≥ 60 分钟", "停留 ≥ 120 分钟"])
ax.set_yticks(range(3), ["离西塔 300 米内", "离西塔 500 米内", "离西塔 1 公里内"])
ax.tick_params(length=0)
ax.grid(False)
for s_ in ax.spines.values():
    s_.set_visible(False)
lo_, hi_ = mat.values.min(), mat.values.max()
title = f"同样叫“西塔到访人数”，口径不同，结果在 {lo_:,} 到 {hi_:,} 人之间，相差 {hi_ / lo_:.1f} 倍"
viz.finish(fig, title,
           "2018-10-03 全天，按“离西塔多近”和“待多久”两个口径统计的到访人数；黑框为本项目采用的口径；口径：dwd_signal、dws_stay",
           SOURCE, "D8_02_到访人数口径对比.png")

In [ ]:
# 对账
assert mat.loc["500 米", "任意一条信令"] == 4_847 and 3_590 <= mat.loc["500 米", "停留 ≥ 30 分钟"] <= 3_620
print(f"✅ 对账通过：500 米内任意信令 4,847 人；500 米内停留 ≥ 30 分钟 {mat.loc['500 米', '停留 ≥ 30 分钟']:,} 人")

In [ ]:
from signal_mysql import database as mysql
# ════ 删失（censoring）════
# 这一格在做什么：看有多少人在数据截止（23:59）时还在西塔。
#
# 这些人的"停留时长"我们只知道**下限**——他们 23:59 还在，之后待到几点无从观测。
# 统计学里把这种"只知道大于某个值"的观测叫**右删失**。
# 处理方式不是丢掉也不是补一个猜测值，而是**在表述上诚实**：
#   写"中位至少 2.7 小时"，不写"中位 2.7 小时"。
#
# 讲台提示：这类问题在企业里到处都是——用户还没流失、订单还没完成、设备还没坏。
# 认出它、并在结论里说清楚，比会算任何模型都更重要。
v = day[day["is_evening_visitor"]].copy()
v["arrive_hour"] = np.floor(v["xita_arrive_hour"]).astype(int)
cens = v[v["arrive_hour"].between(17, 22)].groupby("arrive_hour")["xita_after_23"].mean()
cens_all = v["xita_after_23"].mean()
mysql.write_frame(cens.rename("still_after_23_share").reset_index(), "ads_xita_censoring", con, if_exists="replace", index=False)
fig, ax = plt.subplots(figsize=(10, 5.6))
bars = ax.bar([f"{h} 点" for h in cens.index], cens.values, color=MAIN, width=0.6)
ax.bar_label(bars, labels=[f"{x:.0%}" for x in cens.values], padding=4, color=INK)
ax.axhline(cens_all, color=MUTED, lw=1, ls=(0, (4, 3)))
ax.text(-0.45, cens_all, f"全部傍晚访客 {cens_all:.0%}", va="bottom", color=MUTED, fontsize=9.5)
ax.set_ylim(0, 1)
ax.set_yticks([]); ax.grid(False)
ax.set_xlabel("首次到达西塔的时刻", color=MUTED)
title = f"{cens_all:.0%} 的傍晚访客 23 点后仍在西塔，越晚到的越多：他们的停留时长只是下限"
viz.finish(fig, title,
           "按首次到达时刻分组，23:00 以后仍有西塔停留的访客占比；数据截止 23:59，之后何时离开无法观测；口径：dws_user_day",
           SOURCE, "D8_03_停留时长删失.png")

In [ ]:
# 这一格在做什么：把今天用到的每一个指标，连同它的口径和参数，写成一张表。
#
# 为什么要有口径表：8.5 已经证明"同一个指标名可以差 2.9 倍"。
# 没有这张表，三个月后没人说得清当时用的是哪个口径，报告里的数字就复现不了。
# 企业里这叫"指标字典"或"指标口径管理"，是数据治理最基础也最常被跳过的一环。
#
# ⚠ 现在是手写的，口径一改就要记得同步改这里——这正是不可靠的地方。下面的封装卡 8.2 解决它。
dim_metric = pd.DataFrame([
    ("西塔停留", "在西塔停下来待了一段时间", "停留质心距西塔中心 ≤ 半径，且持续 ≥ 最短时长", "半径 500 米；最短 30 分钟", "dws_stay"),
    ("傍晚访客", "国庆当天傍晚后在西塔待过的人", "有西塔停留，且该停留结束时刻 ≥ 17:00", "17:00", "dws_stay"),
    ("西塔停留时长", "在西塔待了多久", "当天所有西塔停留时长之和；每段按首条到末条信令计", "23 点后仍在者为删失，只是下限", "dws_user_day"),
    ("过夜地", "凌晨待得最久的地方（不等于家）", "与 0:00–6:00 重叠最长且重叠 ≥ 60 分钟的停留", "60 分钟", "dws_night"),
    ("客源距离", "从多远来", "过夜地质心到西塔中心的直线距离", "直线距离，不是路程", "dws_night"),
    ("客群标签", "是哪类人", "按过夜地：不明 / 市外 / 1.5 公里内为周边居民 / 其余为市内来访", "1.5 公里", "dws_user_day"),
    ("客群标签（修正）", "是哪类人，含当天从外地进城的人",
     "当天第一条信令在市域外、之后进入市域，或过夜地在市外：外地来客；其余同客群标签", "—", "dws_user_day.segment_v2"),
    ("出行", "从一个地方到另一个地方", "相邻两个停留质心相距 ≥ 500 米", "500 米", "dws_trip"),
    ("商圈联动率", "西塔访客还去了哪个商圈", "傍晚访客中，当天在该商圈有停留的人的占比", "商圈边界用 v2", "dws_user_day"),
], columns=["metric", "meaning", "definition", "params", "source_table"])
dim_metric["version"] = "v1.1"
dim_metric

In [ ]:
# 由配置渲染的口径表，和上面手写的逐字对比
generated = dims.build_dim_metric(settings)
# D13 起口径表多了 6 条区域级口径：这里只比当天手写的 9 条
gen9 = generated[generated["metric"].isin(dim_metric["metric"])].reset_index(drop=True)
pd.testing.assert_frame_equal(gen9.drop(columns="version"), dim_metric.drop(columns="version"))
db.replace_table(con, generated, "dim_metric")
# 参数一改，口径跟着变：临时把热点半径改成 1000 渲染一次（不写回 yaml）
trial = {**settings, "hotspot": {**settings["hotspot"], "radius_m": 1000}}
print(dims.build_dim_metric(trial).loc[0, "params"])
generated

In [ ]:
# 导出到 data/：明细不进 Git（.gitignore 挡住了 data/ 和 *.csv），也不发给大模型
num_cols = ["n_records", "active_hours", "n_stays", "stay_total_min", "n_trips", "trip_total_km",
            "gyration_km", "night_dist_xita_km", "xita_stay_min"]
day.reset_index().to_csv(config.DATA_DIR / "dws_user_day.csv", index=False, encoding="utf-8-sig")
day[num_cols].corr().round(2)

In [ ]:
# ── 出图 D8_06：箱线图 ───────────────────────────────────────────────────
# 为什么用箱线图：要比较的是**四个客群各自的分布**，不是四个数。
# 只给中位数（四根条形）会掩盖一件事——四类人的分布重叠得很厉害，
# "外地来客活动半径更大"这种直觉在图上根本站不住。箱线图把这个重叠直接画出来。
# 规范落在哪几行：
#   · showfliers=False —— 离群点有上千个，画出来会糊成一片黑，把箱子挤成一条线
#   · set_xscale("log") —— 活动半径从 0.1 到上百公里，线性轴会把 90% 的人挤在最左边
#   · clip(lower=0.1) + 副标题写明"小于 0.1 公里按 0.1 画" ——
#     对数轴画不出 0，做了什么处理必须写出来
#   · 颜色按 SEG_COLOR 固定映射；每个箱子旁边标"中位 X 公里 · N 人"，**把样本量写出来**
# 标题里的数字：med_all（全体中位）、m_["外地来客"] 与 m_["市内来访"]（两类的中位）。
g_ = day[day["gyration_km"].notna()].copy()
g_["r"] = g_["gyration_km"].clip(lower=0.1)
order = ["周边居民", "外地来客", "过夜地不明", "市内来访"]
med_all = g_["gyration_km"].median()
data_ = [g_.loc[g_["segment_v2"] == kk, "r"] for kk in order]
fig, ax = plt.subplots(figsize=(10, 5.6), layout="constrained")
# 多组分布比较用箱线图（手册 4.5），颜色按客群固定映射
bp = ax.boxplot(data_, orientation="horizontal", widths=0.5, patch_artist=True, showfliers=False,
                medianprops=dict(color=INK, lw=2), whiskerprops=dict(color=GRAY_DARK), capprops=dict(color=GRAY_DARK))
for patch, kk in zip(bp["boxes"], order):
    patch.set_facecolor(SEG_COLOR[kk]); patch.set_edgecolor("white")
for i, (kk, d_) in enumerate(zip(order, data_), start=1):
    ax.text(d_.quantile(0.92) * 1.25, i, f"中位 {d_.median():.1f} 公里 · {len(d_):,} 人", va="center", fontsize=9.5, color=INK)
ax.axvline(med_all, color=MUTED, ls="--", lw=1)
ax.text(med_all * 1.05, 4.45, f"全体中位 {med_all:.1f} 公里", fontsize=9, color=MUTED)
ax.set_xscale("log"); ax.set_xlim(0.09, 200)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
ax.set_yticks(range(1, 5), order, fontsize=10, color=INK)
ax.grid(axis="x"); ax.grid(axis="y", visible=False)
ax.set_xlabel("回转半径（公里，对数刻度；箱 = 中间 50% 的人，须 = 1.5 倍四分位距）")
m_ = {kk: d_.median() for kk, d_ in zip(order, data_)}
title = f"一半的人一天的活动半径不到 {med_all:.1f} 公里；外地来客（{m_['外地来客']:.1f}）并不比市内来访（{m_['市内来访']:.1f}）大"
viz.finish(fig, title,
           "回转半径只算停留点，进城路上的移动不计入；小于 0.1 公里（含只有 1 个停留点）按 0.1 画；分群用 segment_v2；口径：dws_user_day",
           SOURCE, "D8_06_活动半径.png")

In [ ]:
con.close()